In [19]:
# 📦 1. Imports
import os, time, torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


In [20]:
import kagglehub
# Setup root directory
itsahmad_indoor_scenes_cvpr_2019_path = kagglehub.dataset_download('itsahmad/indoor-scenes-cvpr-2019')
# print('Data source import complete.')
basePath = itsahmad_indoor_scenes_cvpr_2019_path

rootDir = basePath + "/indoorCVPR_09/Images"

In [21]:
# 📂 2. Dataset Loading + Transforms
# rootDir = '/kaggle/input/indoor-scenes-cvpr-2019/indoorCVPR_09/Images'

allData = []
for label in sorted(os.listdir(rootDir)):
    labelDir = os.path.join(rootDir, label)
    if os.path.isdir(labelDir):
        for file in os.listdir(labelDir):
            if file.lower().endswith(('.jpg', '.jpeg', '.png')):
                allData.append((os.path.join(labelDir, file), label))

trainData, tempData = train_test_split(allData, test_size=0.2, stratify=[label for _, label in allData], random_state=42)
valData, testData = train_test_split(tempData, test_size=0.5, stratify=[label for _, label in tempData], random_state=42)

labelNames = sorted(set(label for _, label in allData))
labelToIndex = {label: idx for idx, label in enumerate(labelNames)}

print(f"Train: {len(trainData)}, Val: {len(valData)}, Test: {len(testData)}")


Train: 12496, Val: 1562, Test: 1562


In [22]:
# 📊 3. Dataset + Transforms
mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]

trainTransform = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.3, 0.3, 0.2),
    transforms.RandomAffine(15, shear=10),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

class SceneDataset(Dataset):
    def __init__(self, data, transform):
        self.data = data
        self.transform = transform

    def __getitem__(self, idx):
        imgPath, label = self.data[idx]
        image = Image.open(imgPath).convert('RGB')
        image = self.transform(image)
        return image, labelToIndex[label]

    def __len__(self):
        return len(self.data)

trainLoader = DataLoader(SceneDataset(trainData, trainTransform), batch_size=64, shuffle=True)
valLoader = DataLoader(SceneDataset(valData, transform), batch_size=64)
testLoader = DataLoader(SceneDataset(testData, transform), batch_size=64)

# fine_label_to_idx = {label: idx for idx, label in enumerate(fine_label_names)}

In [23]:
class SEBlock(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels),
            nn.Sigmoid()
        )
    def forward(self, x):
        b, c, _, _ = x.shape
        se = self.pool(x).view(b, c)
        se = self.fc(se).view(b, c, 1, 1)
        return x * se
    
import torch
import torch.nn as nn
import torch.nn.functional as F

class FoxNet(nn.Module):
    def __init__(self, num_classes=67):
        super(FoxNet, self).__init__()

        self.conv1 = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),  # 224x224 → 224x224
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2)  # 112x112
        )

        self.conv2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),  # 112x112
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2)  # 56x56
        )

        self.conv3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),  # 56x56
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2)  # 28x28
        )

        self.conv4 = nn.Sequential(
            nn.Conv2d(128, 256, kernel_size=3, padding=1),  # 28x28
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1)  # 1x1
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.conv4(x)
        return self.classifier(x)
    # foxnet_best_model

In [24]:
# efficient net
class MiniEfficientNet(nn.Module):
    def __init__(self, num_classes=67):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, 32, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(32), nn.SiLU()
        )

        def MBConv(in_c, out_c, expand=6, stride=1):
            hidden = in_c * expand
            return nn.Sequential(
                nn.Conv2d(in_c, hidden, 1, bias=False), nn.BatchNorm2d(hidden), nn.SiLU(),
                nn.Conv2d(hidden, hidden, 3, stride=stride, padding=1, groups=hidden, bias=False),
                nn.BatchNorm2d(hidden), nn.SiLU(),
                SEBlock(hidden),
                nn.Conv2d(hidden, out_c, 1, bias=False), nn.BatchNorm2d(out_c)
            )

        self.blocks = nn.Sequential(
            MBConv(32, 16, expand=1),
            MBConv(16, 24, stride=2),
            MBConv(24, 40, stride=2),
            MBConv(40, 80, stride=2),
            MBConv(80, 112),
            MBConv(112, 192, stride=2),
            MBConv(192, 320)
        )

        self.head = nn.Sequential(
            nn.Conv2d(320, 1280, 1, bias=False), nn.BatchNorm2d(1280), nn.SiLU(),
            nn.AdaptiveAvgPool2d(1)
        )

        self.classifier = nn.Sequential(
            nn.Dropout(0.4),
            nn.Flatten(),
            nn.Linear(1280, num_classes)
        )

    def forward(self, x):
        x = self.stem(x)
        x = self.blocks(x)
        x = self.head(x)
        return self.classifier(x)

In [27]:
import torch
import torch.nn.functional as F
from sklearn.metrics import classification_report
from tqdm import tqdm

# ------------- Setup -------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_classes = 67

# Load your trained models

model_effnet =  MiniEfficientNet().to(device)
model_foxnet = FoxNet(num_classes=num_classes).to(device)

model_effnet.load_state_dict(torch.load("student_best_mixup_2-2.pth", map_location=device))
model_foxnet.load_state_dict(torch.load("foxnet_student_best_mixup_2-2.pth", map_location=device))

# Move to device
model_effnet = model_effnet.to(device)
model_foxnet = model_foxnet.to(device)

# model_effnet = model_effnet.to(device)

model_effnet.eval()
model_foxnet.eval()

# ------------- Ensemble Prediction Function -------------
def soft_ensemble_predict(imgs, model1, model2):
    with torch.no_grad():
        out1 = F.softmax(model1(imgs), dim=1)
        out2 = F.softmax(model2(imgs), dim=1)
        return (out1 + out2) / 2

# ------------- Evaluation Function -------------
def evaluate_ensemble(model1, model2, dataloader, label_names):
    all_preds = []
    all_labels = []

    for imgs, labels in tqdm(dataloader, desc="Evaluating Ensemble"):
        imgs = imgs.to(device)
        labels = labels.to(device)

        outputs = soft_ensemble_predict(imgs, model1, model2)
        preds = outputs.argmax(dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    print("\nClassification Report (Ensemble):")
    df_report = classification_report(all_labels, all_preds, target_names=label_names)
    print(df_report)

idx_to_label = {idx: label for label, idx in labelToIndex.items()}

label_names = [idx_to_label[i] for i in range(num_classes)]
evaluate_ensemble(model_effnet, model_foxnet, valLoader, label_names)


Evaluating Ensemble: 100%|██████████| 25/25 [00:52<00:00,  2.11s/it]


Classification Report (Ensemble):
                     precision    recall  f1-score   support

     airport_inside       0.66      0.62      0.64        61
          artstudio       1.00      0.29      0.44        14
         auditorium       0.60      0.83      0.70        18
             bakery       0.85      0.57      0.69        40
                bar       0.69      0.58      0.63        60
           bathroom       0.83      0.53      0.65        19
            bedroom       0.66      0.59      0.62        66
          bookstore       0.83      0.63      0.72        38
            bowling       0.75      0.86      0.80        21
             buffet       0.77      0.91      0.83        11
             casino       0.75      0.94      0.84        52
      children_room       0.83      0.91      0.87        11
      church_inside       0.71      0.83      0.77        18
          classroom       0.79      0.92      0.85        12
           cloister       0.86      1.00      0.9